# PHASE 3 — CLASSIFICATION


# Day 16 — Classification Metrics


## 1. Learning Objectives
By the end of this notebook, you will be able to:
- Explain why Accuracy is a terrible metric for imbalanced datasets.
- Read and interpret a Confusion Matrix.
- Calculate and explain Precision, Recall, and the F1 Score.
- Use `classification_report` to instantly evaluate a model.


## 2. Prerequisites
- Day 15 (Logistic Regression).


## 3. Concept: The Accuracy Trap
Yesterday we used **Accuracy** (`Correct / Total`). 
Imagine a dataset of 1,000 credit card transactions where only 1 transaction is Fraud (1) and 999 are Normal (0). This is called an **Imbalanced Dataset**.

If you build a "dumb" model that simply hardcodes `return 0` for every single transaction, it will correctly identify all 999 normal transactions and miss the 1 fraud. 
Its Accuracy is `999 / 1000 = 99.9%`!

The bank just deployed a 99.9% accurate model that catches absolutely zero fraud. Accuracy is a dangerous trap.


## 4. The Confusion Matrix
To understand what our model is actually doing, we break its predictions into 4 buckets:
1. **True Positives (TP)**: Model predicted Fraud, and it WAS Fraud. (Good!)
2. **True Negatives (TN)**: Model predicted Normal, and it WAS Normal. (Good!)
3. **False Positives (FP)**: Model predicted Fraud, but it was Normal. (Annoying - Customer gets angry SMS).
4. **False Negatives (FN)**: Model predicted Normal, but it WAS Fraud. (Catastrophic - Money is stolen).


## 5. Precision vs Recall
From the Confusion Matrix, we calculate two critical metrics:

**Precision**: Out of all the times the model *yelled* "Fraud!", how many times was it actually right?
$$ Precision = \frac{TP}{TP + FP} $$
*High Precision means you don't cry wolf. You rarely annoy customers with false alarms.*

**Recall (Sensitivity)**: Out of all the *actual* Frauds that happened, how many did the model catch?
$$ Recall = \frac{TP}{TP + FN} $$
*High Recall means you catch almost all the bad guys, even if you accidentally annoy some normal customers in the process.*


## 6. The F1 Score
You cannot have 100% Precision and 100% Recall (unless the model is perfect). 
If you want to catch every fraud (High Recall), you have to lower your decision boundary to 0.1, which will flag tons of normal transactions (Low Precision).

The **F1 Score** is the harmonic mean of Precision and Recall. It is a single number that punishes extreme imbalances. If your Precision is 0.99 but your Recall is 0.01, your F1 Score will be terrible (unlike Accuracy).


## 7. Scikit-learn API
```python
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, classification_report
```


## 8. Simple Example
Let's generate a highly imbalanced dataset representing a rare disease (10 sick patients out of 100).


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, classification_report

# 1. Generate Imbalanced Data
np.random.seed(42)
X = np.random.rand(100, 2)
y = np.zeros(100)
y[:10] = 1 # Only 10 positive cases (10% prevalence)
np.random.shuffle(y)

# Make the positive cases somewhat distinct but overlapping
X[y == 1] += 0.5 

# 2. Train Model
model = LogisticRegression()
model.fit(X, y)
y_pred = model.predict(X)


## 9. Code Walkthrough
- We created 100 patients. 90 are Healthy (0), 10 are Sick (1).
- We trained a standard `LogisticRegression` on it and generated predictions.


## 10. Experiment
Let's calculate the Accuracy, Confusion Matrix, and our new metrics.


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print(f'Accuracy:  {accuracy_score(y, y_pred):.2f}')
print(f'Precision: {precision_score(y, y_pred):.2f}')
print(f'Recall:    {recall_score(y, y_pred):.2f}')
print(f'F1 Score:  {f1_score(y, y_pred):.2f}\n')

print('Confusion Matrix:')
cm = confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Healthy (0)', 'Sick (1)'])
disp.plot(cmap='Blues')
plt.show()


> Look at the matrix! The model got an Accuracy of ~90%... but it only caught 2 out of the 10 sick patients (Recall = 0.20)! It missed 8 sick people (False Negatives). This model is dangerous, despite its high accuracy.


## 11. Prediction Exercise
Read the following code, but **DO NOT RUN IT YET**.


In [ ]:
y_true_mock = [1, 1, 1, 0, 0]
y_pred_dumb = [0, 0, 0, 0, 0] # A dumb model that just says 0 to everything


> **Question:** What is the Recall of this dumb model? What is its Precision?

**Think before running the next cell!**


In [ ]:
print('Recall of dumb model:   ', recall_score(y_true_mock, y_pred_dumb, zero_division=0))
print('Precision of dumb model:', precision_score(y_true_mock, y_pred_dumb, zero_division=0))
print('\nWhy? It caught 0 out of the 3 real positives (Recall = 0/3).')
print('It never predicted 1, so Precision is mathematically 0/0 (Scikit-learn defaults to 0).')


## 12. Coding Exercise
Scikit-learn provides a magical function called `classification_report` that calculates all of this instantly for both classes. 
Pass `y` and `y_pred` into `classification_report()` and print the result.


In [ ]:
# YOUR CODE HERE
report = classification_report(y, y_pred)
print(report)
print('\nNotice how Class 0 (Healthy) has near perfect scores, but Class 1 (Sick) has terrible scores. This is exactly what the F1 score reveals!')


## 13. Debugging Challenge
A Machine Learning engineer at a hospital is trying to maximize Recall because they don't want to miss a single tumor. They set the probability threshold to `0.0001`. They achieved 100% Recall! 
However, the doctors are furious and refuse to use the software. Why?


In [ ]:
# Buggy business logic
probs = model.predict_proba(X)[:, 1]
extreme_recall_preds = (probs > 0.0001).astype(int)
print('Recall:   ', recall_score(y, extreme_recall_preds))
print('Precision:', precision_score(y, extreme_recall_preds))


> **Hint:** If you set the threshold to 0.0001, the model will classify almost EVERYONE as having a tumor. 
> Recall hits 100%, but Precision plummets to ~10%. The doctors are furious because the model is screaming "TUMOR!" at 90 healthy people, forcing them to do unnecessary, expensive, and stressful biopsies on healthy patients. 
> **Rule:** There is always a tradeoff between Precision and Recall.


## 14. Model Evaluation (Tradeoff)
You can mathematically slide your Precision and Recall up and down by changing the decision boundary threshold. 
- **Increase Threshold (e.g., 0.9)**: Precision goes up, Recall goes down. (Conservative model).
- **Decrease Threshold (e.g., 0.1)**: Recall goes up, Precision goes down. (Aggressive model).


## 15. Real-World Example
- **YouTube Recommendations**: Google wants to recommend a video you'll click. If they recommend a video you don't like (False Positive), you just ignore it. No big deal. But they want to make sure out of the 5 videos they show, all 5 are highly relevant. They optimize for **High Precision**.
- **Self-Driving Cars**: The car's camera thinks a shadow *might* be a pedestrian. A False Positive means the car slams on the brakes for a shadow (annoying). A False Negative means the car runs over a human (fatal). Tesla optimizes for **High Recall**.


## 16. Mini Project
Write a loop that calculates the F1 score for thresholds: `[0.2, 0.5, 0.8]`. Which threshold gives the highest F1 score for our sick patients?


In [ ]:
thresholds = [0.2, 0.5, 0.8]
for t in thresholds:
    custom_p = (probs > t).astype(int)
    f1 = f1_score(y, custom_p)
    print(f'Threshold {t:.1f} -> F1 Score: {f1:.2f}')

print('\nDropping the threshold to 0.2 made the model slightly more aggressive, which drastically improved its F1 score on this imbalanced dataset!')


## 17. Common Mistakes
- **Using Accuracy on Imbalanced Data**: The #1 mistake junior data scientists make. Always check `value_counts()` on your target variable first!
- **Confusing Precision and Recall**: 
  - Precision = "When you *say* positive, are you right?"
  - Recall = "Did you *find* all the positives?"


## 18. Interview Questions
- **Beginner**: Why is Accuracy a bad metric for detecting credit card fraud?
- **Intermediate**: Explain the difference between a False Positive and a False Negative.
- **Advanced**: If your spam filter is putting important emails from your boss into the Spam folder, is your model suffering from low Precision or low Recall? (Answer: Low Precision. It predicted Spam (Positive) when it was actually Normal (Negative). This is a False Positive).


## 19. Knowledge Check
- What metric is the harmonic mean of Precision and Recall? (F1 Score)
- What function prints all metrics for all classes instantly? (`classification_report`)


## 20. Summary
- **Accuracy** is dangerously misleading on imbalanced datasets.
- **Confusion Matrix** shows TP, TN, FP, FN.
- **Precision**: Focuses on minimizing False Positives (Don't cry wolf).
- **Recall**: Focuses on minimizing False Negatives (Don't miss the bad guys).
- **F1 Score**: Balances both.
- Use `classification_report` for a complete breakdown.


## 21. Homework
Load the `load_wine` dataset. It has 3 classes. Train a Logistic Regression model and use `classification_report`. Notice how the report beautifully handles multi-class classification by providing Precision and Recall for *each* of the 3 wine types separately!
